## Import Files

In [ ]:
import yaml
from mstr_robotics._paths import REPO_ROOT, CONFIG_DIR, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
import pandas as pd
import numpy as np
import json
from mstr_robotics.mstr_classes import get_conn
from mstr_robotics.osi_exporter import export_dashboard
from mstr_robotics.dossier import DossReadOut, DossReadOutDet
with open(USER_CONFIG, 'r', encoding='utf-8') as openfile:
    user_d = yaml.safe_load(openfile)

with open(OSI_SCHEMA, 'r', encoding='utf-8') as openfile:
    osi_dashboard_schema_d = json.load(openfile)

In [ ]:
from mstr_robotics._paths import REPO_ROOT, CONFIG_DIR, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
conn_params = user_d["conn_params"]
conn = get_conn(**conn_params)
conn.headers['Content-type'] = "application/json"

i_doss_read_out     = DossReadOut()
i_doss_read_out_det = DossReadOutDet()
conn.select_project(user_d["conn_params"]["project_id"])

#object ids are maintained in config/jupyter_objects_d.yml
with open(CONFIG_DIR / "jupyter_objects_d.yml", "r") as openfile:
    jupyter_objects_d = yaml.safe_load(openfile)
nb_d = jupyter_objects_d["jup_osi_file_generator"]

dossier_id_l = nb_d["misc"]["dossier_id_l"]

for doss_id in dossier_id_l:
    osi_d       = export_dashboard.build_osi_semantic_model(conn, doss_id)
    doss_hier_l = i_doss_read_out_det.run_read_out_doss_hier_det(conn, [doss_id])
    filt_sel_d  = i_doss_read_out.run_read_out_doss_filt_sel(conn, [doss_id])

    osi_d["dashboards"] = export_dashboard.build_osi_dashboard(
        doss_hier_l=doss_hier_l,
        filt_sel_d=filt_sel_d,
        semantic_model_name=doss_id,   # ID matches semantic_model name
    )

    out_path = str(OSI_FILES / f"osi_dashboard_{doss_id}.yml")
    osi_yaml=export_dashboard.osi_yaml_dump(osi_d, out_path)
    print(f"Written → {out_path}")


osi_d

In [ ]:
from mstr_robotics._paths import REPO_ROOT, CONFIG_DIR, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
ai_context_d={}
ai_sys_dashbaord_prp=""
ai_sys_dashbaord_prp += "Within the dashboard you find URI to Onotologies of Brands and Artists"
ai_sys_dashbaord_prp += "Please consider this information when answering the questions of the user"


#ai_context_d["dashboard_prp"] = ai_sys_dashbaord_prp
ai_context_d["rag_files"] = [str(OSI_DASHBOARD_CONTEXT / "*")]
ai_context_d["grid_ontologies"] = []
osi_d=export_dashboard.add_ai_context(
    osi_d      = osi_d,
    path       = "dashboards",
    ai_context = json.dumps(ai_context_d),
    out_path   = str(OSI_FILES / f"osi_dashboard_{doss_id}.yml"),
)


out_path = str(OSI_FILES / f"osi_dashboard_{doss_id}.yml")
osi_yaml=export_dashboard.osi_yaml_dump(osi_d, out_path)
print(f"Written → {out_path}")


osi_d